<a href="https://colab.research.google.com/github/mayafetzer/LiBatteryExtraction/blob/main/Copy_of_Final_ALNUR_CODE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from google.colab import drive
drive.mount('/content/drive')

warnings.filterwarnings('ignore')


FILE_PATH       = '/content/drive/MyDrive/final/Final_combined.xlsx'
MODEL_SAVE_PATH = '/content/drive/MyDrive/final/finaltrial.pkl'
TARGET_COLUMNS  = ['Li', 'Co', 'Mn', 'Ni']

NUMERIC_FEATURES = [
    'Li in feed  %', 'Co in feed %', 'Mn in feed  %', 'Ni in feed %',
    'Concentration, M', 'Concentration %', 'Time,min  ', 'Temperature, C',
]

CATEGORICAL_FEATURES = ['Leaching agent ', 'Type of reducing agent ']



def load_and_clean(file_path: str) -> pd.DataFrame:
    """Load the Excel file and apply standard cleaning."""
    print(f"Loading data from '{file_path}'...")
    df = pd.read_excel(file_path)
    print(f"  Loaded {len(df)} rows, {df.shape[1]} columns.")

    for metal in TARGET_COLUMNS:
        df[metal] = pd.to_numeric(df[metal], errors='coerce')
        df = df[(df[metal] <= 100) | (df[metal].isna())]

    for col in CATEGORICAL_FEATURES:
        df[col] = (df[col]
                   .astype(str)
                   .str.strip()
                   .str.title()
                   .replace(['None', 'None_Used', 'Nan', 'nan'], 'Unknown'))

    print(f"  After cleaning: {len(df)} rows remain.")
    return df

def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:

    df  = df.copy()
    eps = 1e-6

    df['Time_x_Temp']       = df['Time,min  ']       * df['Temperature, C']
    df['Leach_Conc_x_Time'] = df['Concentration, M'] * df['Time,min  ']
    df['Temp_Squared']       = df['Temperature, C']  ** 2
    df['Time_Squared']       = df['Time,min  ']      ** 2
    df['Acid_to_Reducer']    = df['Concentration, M'] / (df['Concentration %'] + eps)

    return df

ENGINEERED_FEATURES = [
    'Time_x_Temp', 'Leach_Conc_x_Time',
    'Temp_Squared', 'Time_Squared', 'Acid_to_Reducer',
]



def build_preprocessor(all_numeric_features: list,
                        categorical_features: list):

    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
    ])

    categorical_pipe = Pipeline([
        ('onehot', OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=False
        )),
    ])

    preprocessor = ColumnTransformer([
        ('num', numeric_pipe,     all_numeric_features),
        ('cat', categorical_pipe, categorical_features),
    ])

    return preprocessor


def train_global_model(X_train: pd.DataFrame,
                       y_train: pd.DataFrame,
                       preprocessor) -> dict:

    print("\nFitting preprocessor on all training X...")
    X_train_proc = preprocessor.fit_transform(X_train)
    print(f"  Transformed feature matrix shape: {X_train_proc.shape}")

    base_params = dict(
        n_estimators     = 300,
        max_depth        = 4,
        learning_rate    = 0.05,
        subsample        = 0.8,
        min_samples_leaf = 5,
        random_state     = 42,
    )

    print("\nTraining (one GBM per metal, NaN rows excluded per metal)...")
    models = {}

    for metal in y_train.columns:
        valid_mask = y_train[metal].notna().values
        n_valid    = valid_mask.sum()

        if n_valid < 10:
            print(f"  {metal:>3s}: only {n_valid} valid training rows — skipping.")
            models[metal] = None
            continue

        X_metal = X_train_proc[valid_mask]
        y_metal = y_train[metal].values[valid_mask]

        gbm = GradientBoostingRegressor(**base_params)
        gbm.fit(X_metal, y_metal)
        models[metal] = gbm
        print(f"  {metal:>3s}: trained on {n_valid:,} / {len(y_train):,} rows.")

    print("  Training complete.")
    return models


def evaluate(models: dict,
             preprocessor,
             X_test: pd.DataFrame,
             y_test: pd.DataFrame,
             target_columns: list):

    X_test_proc = preprocessor.transform(X_test)

    print("\n" + "=" * 60)
    print("GLOBAL MODEL — TEST SET EVALUATION")
    print("=" * 60)

    all_true, all_pred = [], []

    for metal in target_columns:
        gbm = models.get(metal)
        if gbm is None:
            print(f"  {metal:>3s}: no model trained.")
            continue

        valid_mask = y_test[metal].notna().values
        n_valid    = valid_mask.sum()

        if n_valid == 0:
            print(f"  {metal:>3s}: no valid test rows.")
            continue

        true_vals = y_test[metal].values[valid_mask]
        pred_vals = gbm.predict(X_test_proc[valid_mask])

        r2   = r2_score(true_vals, pred_vals)
        mae  = mean_absolute_error(true_vals, pred_vals)
        rmse = np.sqrt(mean_squared_error(true_vals, pred_vals))

        print(f"  {metal:>3s}  |  R² = {r2:+.4f}  |  MAE = {mae:.2f} %  "
              f"|  RMSE = {rmse:.2f} %  |  n = {n_valid}")

        all_true.extend(true_vals)
        all_pred.extend(pred_vals)

    print("-" * 60)
    if all_true:
        print(f"  ALL  |  R² = {r2_score(all_true, all_pred):+.4f}"
              f"  |  MAE = {mean_absolute_error(all_true, all_pred):.2f} %"
              f"  |  RMSE = {np.sqrt(mean_squared_error(all_true, all_pred)):.2f} %"
              f"  |  n = {len(all_true)}")
    print("=" * 60)


def cross_validate_global(df: pd.DataFrame,
                           all_numeric_features: list,
                           categorical_features: list,
                           target_columns: list,
                           n_splits: int = 5):

    print(f"\nRunning {n_splits}-fold cross-validation...")

    feature_cols = all_numeric_features + categorical_features
    X = df[feature_cols]
    y = df[target_columns]

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    cv_r2  = {m: [] for m in target_columns}
    cv_mae = {m: [] for m in target_columns}

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        prep        = build_preprocessor(all_numeric_features, categorical_features)
        fold_models = train_global_model(X_tr, y_tr, prep)
        X_val_proc  = prep.transform(X_val)

        for metal in target_columns:
            gbm = fold_models.get(metal)
            if gbm is None:
                continue

            valid_mask = y_val[metal].notna().values
            if valid_mask.sum() < 2:
                continue

            true_vals = y_val[metal].values[valid_mask]
            pred_vals = gbm.predict(X_val_proc[valid_mask])

            cv_r2[metal].append(r2_score(true_vals, pred_vals))
            cv_mae[metal].append(mean_absolute_error(true_vals, pred_vals))

        print(f"  Fold {fold} done.")

    print("\n" + "=" * 60)
    print(f"CROSS-VALIDATION RESULTS ({n_splits}-fold)")
    print("=" * 60)
    for metal in target_columns:
        if cv_r2[metal]:
            print(f"  {metal:>3s}  |  R² = {np.mean(cv_r2[metal]):.4f} "
                  f"± {np.std(cv_r2[metal]):.4f}"
                  f"  |  MAE = {np.mean(cv_mae[metal]):.2f} "
                  f"± {np.std(cv_mae[metal]):.2f} %")
    print("=" * 60)


def plot_feature_importances(models: dict,
                              preprocessor,
                              all_numeric_features: list,
                              categorical_features: list,
                              target_columns: list,
                              top_n: int = 20):
    """Average feature importances across all trained metal GBMs."""
    ohe       = preprocessor.named_transformers_['cat']['onehot']
    ohe_names = list(ohe.get_feature_names_out(categorical_features))
    feat_names = all_numeric_features + ohe_names

    trained = [models[m] for m in target_columns if models.get(m) is not None]
    if not trained:
        print("No trained models available for importance plot.")
        return

    importances = np.mean([gbm.feature_importances_ for gbm in trained], axis=0)

    fi_df = (pd.DataFrame({'feature': feat_names, 'importance': importances})
               .sort_values('importance', ascending=False)
               .head(top_n))

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(fi_df['feature'][::-1], fi_df['importance'][::-1], color='steelblue')
    ax.set_xlabel('Mean Feature Importance (averaged across all metal models)')
    ax.set_title(f'Top {top_n} Features — Global Leaching Model')
    plt.tight_layout()
    plt.savefig('feature_importances.png', dpi=150)
    print("\nFeature importance plot saved to 'feature_importances.png'")
    plt.show()


def prepare_single_prediction(li_feed, co_feed, mn_feed, ni_feed,
                               leaching_agent, leaching_conc_M,
                               reducing_agent, reducing_conc_pct,
                               time_min, temp_C,
                               all_numeric_features, categorical_features):

    eps = 1e-6
    row = {
        'Li in feed  %'           : li_feed,
        'Co in feed %'            : co_feed,
        'Mn in feed  %'           : mn_feed,
        'Ni in feed %'            : ni_feed,
        'Concentration, M'        : leaching_conc_M,
        'Concentration %'         : reducing_conc_pct,
        'Time,min  '              : time_min,
        'Temperature, C'          : temp_C,
        'Leaching agent '         : leaching_agent.strip().title(),
        'Type of reducing agent ' : reducing_agent.strip().title(),
        'Time_x_Temp'             : time_min * temp_C,
        'Leach_Conc_x_Time'       : leaching_conc_M * time_min,
        'Temp_Squared'            : temp_C ** 2,
        'Time_Squared'            : time_min ** 2,
        'Acid_to_Reducer'         : leaching_conc_M / (reducing_conc_pct + eps),
    }
    return pd.DataFrame([row])[all_numeric_features + categorical_features]


def predict(models: dict,
            preprocessor,
            input_df: pd.DataFrame,
            target_columns: list) -> dict:
    X_proc  = preprocessor.transform(input_df)
    results = {}
    for metal in target_columns:
        gbm = models.get(metal)
        if gbm is None:
            results[metal] = None
        else:
            val = gbm.predict(X_proc)[0]
            results[metal] = float(np.clip(val, 0, 100))
    return results



def main():

    df = load_and_clean(FILE_PATH)

    df = add_engineered_features(df)
    all_numeric = NUMERIC_FEATURES + ENGINEERED_FEATURES

    feature_cols = all_numeric + CATEGORICAL_FEATURES
    X = df[feature_cols]
    y = df[TARGET_COLUMNS]

    print(f"\nDataset summary:")
    print(f"  Total samples : {len(X):,}")
    print(f"  Feature count : {len(feature_cols)} "
          f"({len(all_numeric)} numeric + {len(CATEGORICAL_FEATURES)} categorical)")
    print(f"  Target coverage (non-NaN rows):")
    for metal in TARGET_COLUMNS:
        n = y[metal].notna().sum()
        print(f"    {metal}: {n:,} / {len(y):,} rows ({100*n/len(y):.1f}%)")
    print(f"  Unique acid pairs in data:")
    for pair, count in df.groupby(CATEGORICAL_FEATURES).size().items():
        print(f"    {pair[0]} + {pair[1]}: {count} samples")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )
    print(f"\nTrain: {len(X_train):,} samples  |  Test: {len(X_test):,} samples")


    preprocessor = build_preprocessor(all_numeric, CATEGORICAL_FEATURES)
    models       = train_global_model(X_train, y_train, preprocessor)

    evaluate(models, preprocessor, X_test, y_test, TARGET_COLUMNS)

    cross_validate_global(df, all_numeric, CATEGORICAL_FEATURES, TARGET_COLUMNS)

    plot_feature_importances(models, preprocessor, all_numeric,
                             CATEGORICAL_FEATURES, TARGET_COLUMNS, top_n=20)

    artifact = {
        'models'              : models,
        'preprocessor'        : preprocessor,
        'all_numeric_features': all_numeric,
        'categorical_features': CATEGORICAL_FEATURES,
        'target_columns'      : TARGET_COLUMNS,
    }
    joblib.dump(artifact, MODEL_SAVE_PATH)
    print(f"\nModel saved → '{MODEL_SAVE_PATH}'")

    print("\n" + "=" * 60)
    print("EXAMPLE PREDICTION (global model — no acid-pair lookup needed)")
    print("=" * 60)

    example_input = prepare_single_prediction(
        li_feed           = 6.79,
        co_feed           = 17.68,
        mn_feed           = 16.46,
        ni_feed           = 17.58,
        leaching_agent    = 'Lactic Acid',
        leaching_conc_M   = 1.5,
        reducing_agent    = 'Hydrogen Peroxide',
        reducing_conc_pct = 1.5,
        time_min          = 50,
        temp_C            = 60,
        all_numeric_features = all_numeric,
        categorical_features = CATEGORICAL_FEATURES,
    )

    preds = predict(models, preprocessor, example_input, TARGET_COLUMNS)
    print(f"  Leaching agent : Lactic Acid (1.5 M)")
    print(f"  Reducing agent : Hydrogen Peroxide (1.5 %)")
    print(f"  Time / Temp    : 50 min / 60 °C")
    print(f"  Metal feed     : Li={6.79}%, Co={17.68}%, Mn={16.46}%, Ni={17.58}%")
    print(f"\n  Predicted efficiencies:")
    for metal, val in preds.items():
        display = f"{val:.2f} %" if val is not None else "N/A (model not trained)"
        print(f"    {metal}: {display}")
    print("\n  (Reference values: Li=87.77, Co=76.06, Mn=80.21, Ni=75.97)")


def example_load_and_predict():
    artifact = joblib.load('global_leaching_model.pkl')

    models        = artifact['models']
    preprocessor  = artifact['preprocessor']
    all_numeric   = artifact['all_numeric_features']
    categorical   = artifact['categorical_features']
    target_cols   = artifact['target_columns']

    input_df = prepare_single_prediction(
        li_feed           = 5.85,
        co_feed           = 46.5,
        mn_feed           = 3.22,
        ni_feed           = 2.69,
        leaching_agent    = 'Ammonium Bicarbonate',
        leaching_conc_M   = 2.5,
        reducing_agent    = 'Ammonium Sulfite',
        reducing_conc_pct = 5.81,
        time_min          = 90,
        temp_C            = 50,
        all_numeric_features = all_numeric,
        categorical_features = categorical,
    )

    results = predict(models, preprocessor, input_df, target_cols)
    print("Predicted efficiencies:", results)


if __name__ == '__main__':
    main()

Mounted at /content/drive
Loading data from '/content/drive/MyDrive/final/Final_combined.xlsx'...
  Loaded 5940 rows, 14 columns.
  After cleaning: 5940 rows remain.

Dataset summary:
  Total samples : 5,940
  Feature count : 15 (13 numeric + 2 categorical)
  Target coverage (non-NaN rows):
    Li: 5,644 / 5,940 rows (95.0%)
    Co: 5,724 / 5,940 rows (96.4%)
    Mn: 1,726 / 5,940 rows (29.1%)
    Ni: 2,238 / 5,940 rows (37.7%)
  Unique acid pairs in data:
    Ammonium Bicarbonate + Ammonium Sulfite: 138 samples
    Ascorbic Acid + Unknown: 30 samples
    Citric Acid + Hydrogen Peroxide: 354 samples
    Citric Acid + Unknown: 6 samples
    Dl-Malic Acid + Hydrogen Peroxide: 366 samples
    Hydrochloric Acid + Unknown: 984 samples
    Hydrogen Peroxide + Unknown: 120 samples
    Hydroxylammonium Chloride + Unknown: 78 samples
    Iminodiacetic Acid + Ascorbic Acid: 6 samples
    L-Tartaric Acid + Hydrogen Peroxide: 156 samples
    Lactic Acid + Hydrogen Peroxide: 306 samples
    Malic A

In [ ]:

import pandas as pd
import numpy as np
import warnings
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

warnings.filterwarnings('ignore')
INPUT_FILE        = '/content/drive/MyDrive/final/original.xlsx'
OUTPUT_FILE       = '/content/drive/MyDrive/final/final(synthesized).xlsx'
COMBINED_OUTPUT   = '/content/drive/MyDrive/final/Final_combined.xlsx'

TARGET_TOTAL_ROWS = 5000
AUGMENT_MULTIPLIER = 5
NOISE_SCALE       = 0.04
RANDOM_SEED       = 42


FEED_COLS    = ['Li in feed  %', 'Co in feed %', 'Mn in feed  %', 'Ni in feed %']
COND_COLS    = ['Concentration, M', 'Concentration %', 'Time,min  ', 'Temperature, C']
CAT_COLS     = ['Leaching agent ', 'Type of reducing agent ']
TARGET_COLS  = ['Li', 'Co', 'Mn', 'Ni']
ALL_COLS     = FEED_COLS + CAT_COLS + COND_COLS + TARGET_COLS

def standardize_categoricals(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in CAT_COLS:
        df[col] = (df[col].astype(str).str.strip().str.title()
                   .replace(['None', 'None_Used', 'Nan', 'nan', 'Unknown'], 'Unknown'))
    return df


def build_target_model(df: pd.DataFrame):
    """
    Train one GBM per target metal on the full original dataset.
    These models predict realistic efficiency values for synthetic conditions
    before adding calibrated noise.
    """
    numeric_features = FEED_COLS + COND_COLS

    numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median'))])
    categorical_pipe = Pipeline([('ohe', OneHotEncoder(
        handle_unknown='ignore', sparse_output=False))])

    preprocessor = ColumnTransformer([
        ('num', numeric_pipe,      numeric_features),
        ('cat', categorical_pipe,  CAT_COLS),
    ])

    X = df[numeric_features + CAT_COLS]
    preprocessor.fit(X)

    models = {}
    for metal in TARGET_COLS:
        mask  = df[metal].notna()
        n     = mask.sum()
        if n < 5:
            models[metal] = None
            continue
        X_m   = preprocessor.transform(X[mask])
        y_m   = df[metal][mask].values
        gbm   = GradientBoostingRegressor(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, min_samples_leaf=3, random_state=RANDOM_SEED
        )
        gbm.fit(X_m, y_m)
        models[metal] = gbm

    return preprocessor, models


def measure_noise_per_group(df: pd.DataFrame) -> dict:
    """
    Estimate the realistic noise level for each (metal, group) combination
    from the original data. Uses the median absolute deviation of residuals
    after fitting a local mean surface to the data.
    Returns a dict: {(metal, leaching, reducing): noise_std}
    """
    noise = {}
    groups = df.groupby(CAT_COLS)
    for name, grp in groups:
        for metal in TARGET_COLS:
            vals = grp[metal].dropna()
            if len(vals) < 3:
                noise[(metal,) + name] = 2.0
                continue
            cond_groups = grp.groupby(COND_COLS, dropna=False)[metal]
            residuals = []
            for _, cg in cond_groups:
                cg = cg.dropna()
                if len(cg) > 1:
                    residuals.extend((cg - cg.mean()).values)
            if len(residuals) >= 3:
                noise_val = np.std(residuals)
            else:
                noise_val = vals.std() * NOISE_SCALE
            noise[(metal,) + name] = max(noise_val, 0.5)  # at least 0.5%

    return noise


def perturb_conditions(group_data: pd.DataFrame,
                        n_samples: int,
                        rng: np.random.Generator) -> pd.DataFrame:

    resampled = group_data[COND_COLS].sample(
        n=n_samples, replace=True, random_state=int(rng.integers(0, 1e6))
    ).reset_index(drop=True)

    group_has_no_reducer = (group_data[CAT_COLS[1]].iloc[0] == 'Unknown')

    for col in COND_COLS:
        col_range = group_data[col].max() - group_data[col].min()

        # No reducer → Concentration % is always 0, never perturb
        if col == 'Concentration %' and group_has_no_reducer:
            resampled[col] = 0.0
            continue

        if col_range < 1e-9:
            continue

        sigma = col_range * 0.05   # 5% of range
        noise = rng.normal(0, sigma, size=n_samples)
        resampled[col] = (resampled[col] + noise).clip(
            group_data[col].min(), group_data[col].max()
        )

        if col in ['Time,min  ', 'Temperature, C']:
            resampled[col] = resampled[col].round().astype(int)

    return resampled


def generate_targets(synth_conditions: pd.DataFrame,
                     feed_row: pd.Series,
                     group_name: tuple,
                     preprocessor,
                     models: dict,
                     noise_lookup: dict,
                     orig_group: pd.DataFrame,
                     rng: np.random.Generator) -> pd.DataFrame:
    """
    Predict targets using GBM, then add calibrated noise.
    Enforce [0, 100] bounds hard.
    Preserve NaN pattern: if a metal was never reported for this group,
    keep it NaN in synthetic rows too.
    """
    n = len(synth_conditions)

    X_synth = synth_conditions.copy()
    for col in FEED_COLS:
        X_synth[col] = feed_row[col]
    for col in CAT_COLS:
        X_synth[col] = group_name[CAT_COLS.index(col)]

    X_proc = preprocessor.transform(X_synth[FEED_COLS + COND_COLS + CAT_COLS])

    target_df = pd.DataFrame(index=range(n))

    for metal in TARGET_COLS:
        # If this metal was never observed in this group → keep NaN
        metal_vals = orig_group[metal].dropna()
        if len(metal_vals) == 0:
            target_df[metal] = np.nan
            continue

        gbm = models.get(metal)
        if gbm is None:
            target_df[metal] = np.nan
            continue

        pred = gbm.predict(X_proc)

        key       = (metal,) + group_name
        noise_std = noise_lookup.get(key, 2.0)
        noise     = rng.normal(0, noise_std, size=n)
        noisy     = pred + noise

        noisy = np.clip(noisy, 0.0, 100.0)

        metal_min = max(0.0,   metal_vals.min() - 5.0)
        metal_max = min(100.0, metal_vals.max() + 5.0)
        noisy = np.clip(noisy, metal_min, metal_max)

        target_df[metal] = np.round(noisy, 2)

    return target_df


def synthesize_group(group_name: tuple,
                     orig_group: pd.DataFrame,
                     n_to_generate: int,
                     preprocessor,
                     models: dict,
                     noise_lookup: dict,
                     rng: np.random.Generator) -> pd.DataFrame:
    """
    Synthesize n_to_generate rows for one acid-pair group.
    """
    if len(orig_group) < 2:
        # Too few rows — just replicate with tiny noise
        return orig_group.sample(n=n_to_generate, replace=True,
                                 random_state=int(rng.integers(0, 1e6))
                                 ).reset_index(drop=True)

    feed_combos = orig_group[FEED_COLS].drop_duplicates().reset_index(drop=True)

    n_combos   = len(feed_combos)
    base_n     = n_to_generate // n_combos
    remainder  = n_to_generate  % n_combos

    rows = []
    for i, (_, feed_row) in enumerate(feed_combos.iterrows()):
        n_this = base_n + (1 if i < remainder else 0)
        if n_this == 0:
            continue

        mask = np.ones(len(orig_group), dtype=bool)
        for col in FEED_COLS:
            mask &= (orig_group[col].values == feed_row[col])
        sub_group = orig_group[mask]

        if len(sub_group) < 1:
            sub_group = orig_group

        synth_conds = perturb_conditions(sub_group, n_this, rng)

        synth_targets = generate_targets(
            synth_conds, feed_row, group_name,
            preprocessor, models, noise_lookup, sub_group, rng
        )

        row_df = synth_conds.copy()
        for j, col in enumerate(FEED_COLS):
            row_df[col] = feed_row[col]
        row_df[CAT_COLS[0]] = group_name[0]
        row_df[CAT_COLS[1]] = group_name[1]
        for col in TARGET_COLS:
            row_df[col] = synth_targets[col].values

        rows.append(row_df)

    if not rows:
        return pd.DataFrame(columns=ALL_COLS)

    return pd.concat(rows, ignore_index=True)[ALL_COLS]



def validate_synthetic(orig: pd.DataFrame, synth: pd.DataFrame):
    """Compare key statistics between original and synthetic data."""
    print("\n" + "=" * 65)
    print("VALIDATION: ORIGINAL vs SYNTHETIC STATISTICS")
    print("=" * 65)

    for metal in TARGET_COLS:
        o_vals = orig[metal].dropna()
        s_vals = synth[metal].dropna()
        if len(s_vals) == 0:
            continue

        o_mean, o_std = o_vals.mean(), o_vals.std()
        s_mean, s_std = s_vals.mean(), s_vals.std()
        o_q25, o_q75  = o_vals.quantile(0.25), o_vals.quantile(0.75)
        s_q25, s_q75  = s_vals.quantile(0.25), s_vals.quantile(0.75)
        pct_oob = ((s_vals < 0) | (s_vals > 100)).mean() * 100

        print(f"\n  {metal}:")
        print(f"    Mean  : orig={o_mean:.2f}  synth={s_mean:.2f}  "
              f"(Δ={abs(s_mean-o_mean):.2f})")
        print(f"    Std   : orig={o_std:.2f}   synth={s_std:.2f}  "
              f"(Δ={abs(s_std-o_std):.2f})")
        print(f"    Q25   : orig={o_q25:.2f}   synth={s_q25:.2f}")
        print(f"    Q75   : orig={o_q75:.2f}   synth={s_q75:.2f}")
        print(f"    Out-of-bounds [0,100]: {pct_oob:.2f}%  ← must be 0.00%")

    print("\n  Condition ranges (synthetic must not exceed original):")
    for col in COND_COLS:
        o_min, o_max = orig[col].min(), orig[col].max()
        s_min, s_max = synth[col].min(), synth[col].max()
        ok = "✓" if s_min >= o_min - 0.01 and s_max <= o_max + 0.01 else "✗ VIOLATION"
        print(f"    {col:25s}: orig=[{o_min:.2f},{o_max:.2f}]  "
              f"synth=[{s_min:.2f},{s_max:.2f}]  {ok}")

    print("\n  Per-group mean drift (top 5 worst, should be < 5%):")
    for metal in ['Li', 'Co']:
        orig_m = orig.groupby(CAT_COLS)[metal].mean()
        synth_m = synth.groupby(CAT_COLS)[metal].mean()
        drift = (synth_m - orig_m).abs().dropna().sort_values(ascending=False)
        print(f"    {metal} drift (top 5):")
        for idx, val in drift.head(5).items():
            print(f"      {idx[0]} + {idx[1]}: {val:.2f}%")

    print("=" * 65)



def main():
    rng = np.random.default_rng(RANDOM_SEED)

    print(f"Loading '{INPUT_FILE}'...")
    orig = pd.read_excel(INPUT_FILE)
    orig = standardize_categoricals(orig)
    for metal in TARGET_COLS:
        orig[metal] = pd.to_numeric(orig[metal], errors='coerce')
        orig = orig[(orig[metal] <= 100) | (orig[metal].isna())]
    print(f"  {len(orig)} rows loaded after cleaning.")


    print("\nTraining GBM prediction models on original data...")
    preprocessor, models = build_target_model(orig)
    trained = [m for m in TARGET_COLS if models.get(m) is not None]
    print(f"  Models trained for: {trained}")

    print("Measuring per-group noise levels...")
    noise_lookup = measure_noise_per_group(orig)

    groups      = orig.groupby(CAT_COLS)
    group_names = list(groups.groups.keys())
    group_sizes = {name: len(groups.get_group(name)) for name in group_names}
    total_orig  = sum(group_sizes.values())


    n_per_group = {}
    for name, size in group_sizes.items():
        proportion    = size / total_orig
        n_synthetic   = max(1, int(round(proportion * TARGET_TOTAL_ROWS)))
        n_per_group[name] = n_synthetic

    print(f"\nGenerating ~{TARGET_TOTAL_ROWS} synthetic rows across "
          f"{len(group_names)} groups...")

    all_synthetic = []

    for name in group_names:
        orig_group = groups.get_group(name).reset_index(drop=True)
        n_needed   = n_per_group[name]

        synth_group = synthesize_group(
            name, orig_group, n_needed,
            preprocessor, models, noise_lookup, rng
        )
        all_synthetic.append(synth_group)
        print(f"  ✓ {name[0]} + {name[1]}: {n_needed} synthetic rows")

    synthetic_df = pd.concat(all_synthetic, ignore_index=True)
    synthetic_df = synthetic_df[ALL_COLS]


    validate_synthetic(orig, synthetic_df)

    synthetic_df.to_excel(OUTPUT_FILE, index=False)
    print(f"\nSynthetic-only dataset saved → '{OUTPUT_FILE}'  "
          f"({len(synthetic_df)} rows)")

    print(f"\nBuilding combined dataset "
          f"(original + {AUGMENT_MULTIPLIER}× augmented per group)...")

    augmented_parts = []
    for name in group_names:
        orig_group  = groups.get_group(name).reset_index(drop=True)
        n_augmented = len(orig_group) * AUGMENT_MULTIPLIER

        synth_group = synthesize_group(
            name, orig_group, n_augmented,
            preprocessor, models, noise_lookup, rng
        )
        augmented_parts.append(synth_group)

    augmented_df = pd.concat(augmented_parts, ignore_index=True)
    combined_df  = pd.concat([orig, augmented_df], ignore_index=True)
    combined_df  = combined_df[ALL_COLS]

    combined_df.to_excel(COMBINED_OUTPUT, index=False)
    print(f"Combined dataset saved → '{COMBINED_OUTPUT}'  "
          f"({len(orig)} original + {len(augmented_df)} synthetic = "
          f"{len(combined_df)} total rows)")

    # ── 9. Final summary
    print("\n" + "=" * 65)
    print("SUMMARY")
    print("=" * 65)
    print(f"  Original rows       : {len(orig):,}")
    print(f"  Synthetic-only rows : {len(synthetic_df):,}  → '{OUTPUT_FILE}'")
    print(f"  Combined rows       : {len(combined_df):,}  → '{COMBINED_OUTPUT}'")
    print(f"  Out-of-bounds [0,100] in combined: "
          f"{sum((combined_df[m].dropna() > 100).sum() + (combined_df[m].dropna() < 0).sum() for m in TARGET_COLS)} rows")
    print("=" * 65)
    print("\nUse the COMBINED file as input to global_leaching_model.py")


if __name__ == '__main__':
    main()

Loading '/content/drive/MyDrive/final/original.xlsx'...
  990 rows loaded after cleaning.

Training GBM prediction models on original data...
  Models trained for: ['Li', 'Co', 'Mn', 'Ni']
Measuring per-group noise levels...

Generating ~5000 synthetic rows across 24 groups...
  ✓ Ammonium Bicarbonate + Ammonium Sulfite: 116 synthetic rows
  ✓ Ascorbic Acid + Unknown: 25 synthetic rows
  ✓ Citric Acid + Hydrogen Peroxide: 298 synthetic rows
  ✓ Citric Acid + Unknown: 5 synthetic rows
  ✓ Dl-Malic Acid + Hydrogen Peroxide: 308 synthetic rows
  ✓ Hydrochloric Acid + Unknown: 828 synthetic rows
  ✓ Hydrogen Peroxide + Unknown: 101 synthetic rows
  ✓ Hydroxylammonium Chloride + Unknown: 66 synthetic rows
  ✓ Iminodiacetic Acid + Ascorbic Acid: 5 synthetic rows
  ✓ L-Tartaric Acid + Hydrogen Peroxide: 131 synthetic rows
  ✓ Lactic Acid + Hydrogen Peroxide: 258 synthetic rows
  ✓ Malic Acid + Ascorbic Acid: 25 synthetic rows
  ✓ Nitric Acid + Hydrogen Peroxide: 212 synthetic rows
  ✓ Nitric 